# 15차시 실습 — 내 MVP의 AI 기능을 '평가'한다 (eval suite)

> **day1·day2 연속 실습.** day2에서 만든 내 서비스의 AI 기능을, 이제 **믿을 수 있게** 만듭니다.
> 오늘은 작은 **평가 스위트**를 직접 만듭니다 — **골든셋 → LLM-as-judge 채점 → 통과율 표**.

## 🧪 사용법
- 본 실습은 **Codex CLI**로, 이 노트북은 같은 개념을 **OpenAI API로 즉시 실행**하는 동반 자료입니다.
- 각 단계에 **⌨️ 터미널 Codex** 명령(복붙용)을 함께 적어 두었습니다. 위에서부터 실행하세요.


In [2]:
# ✅ 환경 준비 — 맨 처음 한 번만 (day1 노트북과 동일)
%pip install -q openai
import os, json, getpass
from openai import OpenAI
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY (화면에 안 보임): ")
client = OpenAI(); MODEL = "gpt-4o-mini"
print("준비 완료 ·", MODEL)


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
준비 완료 · gpt-4o-mini


## 오늘의 연속 시나리오

day1: 정해진 동작을 하는 앱을 만들었고, day2: 자유 요청을 처리하는 **에이전트**로 키웠습니다.
day3: 그 에이전트가 **‘되는 것 같다’가 아니라 정말 ‘된다’** 를 증명합니다.

> 아래 `my_service()` 가 **당신 팀 MVP의 AI 기능**을 대표합니다(공통 예시 = 맛집 추천 도우미).
> 구조는 동일하니, STEP 7에서 **내 MVP의 입력·기준으로 바꾸면 그대로 적용**됩니다.


## STEP 1 — 평가 대상: 내 서비스의 AI 기능

평가하려면 먼저 **무엇을 평가할지** 정합니다. `my_service(question)` 가 day2에서 만든 내 기능이라고 생각하세요.

⌨️ **터미널 Codex:** `codex "우리 MVP의 AI 기능을 my_service(question)->str 형태로 감싸줘"`


In [3]:
# day2에서 만든 내 서비스의 AI 기능 (공통 예시: 맛집 추천 도우미)
# 실제로는 day2의 에이전트/RAG/체인을 여기에 연결하면 됩니다.
def my_service(question: str) -> str:
    sys = "너는 맛집 추천 도우미다. 한국어로, 간결하게 추천 이유와 함께 답하라."
    r = client.chat.completions.create(model=MODEL, messages=[
        {"role":"system","content":sys},
        {"role":"user","content":question}])
    return r.choices[0].message.content

print(my_service("강남에서 매운 점심 한 곳만 추천해줘"))

강남역 근처의 "직화구이떡볶이"를 추천합니다. 매콤한 떡볶이와 다양한 해산물, 고기를 함께 구워 먹을 수 있어 만족도가 높습니다. 점심 메뉴도 다양하고 가성비가 좋습니다!


## STEP 2 — 골든셋 정의 (입력 → 기준)

**골든셋** = 평가의 단일 기준점. 입력과 ‘좋은 답의 기준’을 짝지어 적습니다. **작게 시작(5개)**, 실패 사례를 계속 추가합니다.

⌨️ **터미널 Codex:** `codex "내 기능의 대표 질문 5개와 합격 기준을 골든셋으로 만들어줘"`


In [4]:
# 입력(question) → 기준(criteria). 정답 한 줄이 아니라 '무엇을 충족해야 하는가'.
GOLDEN = [
  {"q":"강남에서 매운 점심 추천해줘",          "criteria":"강남의 매운 음식점을 1곳 이상 구체적으로 추천하고 이유를 댄다"},
  {"q":"2만원 이하로 먹을 만한 곳 알려줘",       "criteria":"예산(2만원 이하)을 고려한 추천을 제시한다"},
  {"q":"홍대에서 순한 음식 먹고 싶어",          "criteria":"홍대의 순한(맵지 않은) 메뉴/식당을 추천한다"},
  {"q":"채식 가능한 곳 있어?",                  "criteria":"채식/비건 관점을 반영한 답을 한다"},
  {"q":"오늘 날씨 어때?",                       "criteria":"맛집 추천 범위를 벗어난 질문임을 인지하고 정중히 안내한다(환각 금지)"},
]
print(f"골든셋 {len(GOLDEN)}개 준비")

골든셋 5개 준비


## STEP 3 — LLM-as-judge: 다른 LLM에게 채점을 맡긴다

심판 모델에게 **루브릭 + 입력 + 답**을 주고 **1~5점 + 근거(JSON)** 를 받습니다. `score >= 4` 면 통과로 봅니다.

⌨️ **터미널 Codex:** `codex "루브릭으로 1~5점과 근거를 매기는 LLM-as-judge 함수를 만들어줘"`


In [5]:
def judge(question: str, answer: str, criteria: str) -> dict:
    rubric = ("너는 엄격한 채점관이다. 아래 기준을 답이 얼마나 충족하는지 1~5점으로 매겨라. "
              "반드시 JSON으로만 출력: {\"score\": 정수1~5, \"reason\": \"한 문장\"}")
    r = client.chat.completions.create(model=MODEL,
        response_format={"type":"json_object"},
        messages=[{"role":"system","content":rubric},
                  {"role":"user","content":f"[질문]{question}\n[기준]{criteria}\n[답]{answer}"}])
    return json.loads(r.choices[0].message.content)

# 한 건 테스트
ans = my_service(GOLDEN[0]["q"])
print(ans, "\n→", judge(GOLDEN[0]["q"], ans, GOLDEN[0]["criteria"]))

강남의 "매운국수"를 추천합니다. 이곳은 다양한 종류의 매운 국수를 제공하며, 특히 비빔국수가 인기가 많습니다. 매콤한 소스와 신선한 재료가 잘 어우러져 식사 후 개운함을 느낄 수 있습니다. 빠르고 간편한 점심에 좋습니다! 
→ {'score': 4, 'reason': '한 곳의 맛집을 추천하였지만, 구체적인 위치나 다른 매운 음식점의 예시가 부족하다.'}


## STEP 4 — 전체 실행 · 통과율 표

골든셋 전체를 돌려 **케이스별 점수·통과여부**를 표로 만들고 **통과율**을 집계합니다. 이게 ‘믿을 근거’입니다.


In [6]:
PASS = 4  # 4점 이상이면 통과

def run_eval(golden):
    rows, passed = [], 0
    for g in golden:
        ans = my_service(g["q"])
        v = judge(g["q"], ans, g["criteria"])
        ok = v["score"] >= PASS; passed += ok
        rows.append((g["q"][:18], v["score"], "✅" if ok else "❌", v["reason"][:30]))
    print(f"{'질문':<20}{'점수':<6}{'통과':<6}근거")
    print("-"*70)
    for q,s,o,why in rows: print(f"{q:<20}{s:<6}{o:<6}{why}")
    rate = passed/len(golden)*100
    print("-"*70); print(f"통과율: {passed}/{len(golden)} = {rate:.0f}%")
    return rate

baseline = run_eval(GOLDEN)

질문                  점수    통과    근거
----------------------------------------------------------------------
강남에서 매운 점심 추천해줘     5     ✅     강남의 매운 음식점인 '청년다방'과 그 이유를 구체적으
2만원 이하로 먹을 만한 곳 알려  5     ✅     모든 추천이 2만원 이하의 예산을 고려한 다양한 옵션을
홍대에서 순한 음식 먹고 싶어    5     ✅     홍대에서 순한 음식에 적합한 식당을 구체적으로 추천하며
채식 가능한 곳 있어?        2     ❌     채식 가능성을 언급했지만 '한우'가 포함돼 있어 비건 
오늘 날씨 어때?           5     ✅     질문에 대해 적절히 인지하고 정중하게 안내하며 추가 질
----------------------------------------------------------------------
통과율: 4/5 = 80%


## STEP 4b — 결정적 체크 1개 (LLM 없이)

모든 걸 LLM이 채점할 필요는 없습니다. **형식·금지어·길이**처럼 `==`로 명확한 것은 **결정적 체크**가 더 싸고 안정적입니다.


In [7]:
def deterministic_check(answer: str) -> bool:
    # 예: 너무 짧지 않고, 사과만 하고 끝나지 않는다
    return len(answer) >= 10 and "죄송" not in answer[:6]

sample = my_service("강남 매운 점심")
print("결정적 체크:", "✅통과" if deterministic_check(sample) else "❌실패")

결정적 체크: ✅통과


## STEP 5 — pass@k: 한 번 통과는 운일 수 있다

같은 케이스를 k번 돌려 **성공률**을 봅니다. 비결정적 모델에서 '가끔 통과'로 숨는 불안정 케이스를 드러냅니다.

In [9]:
K = 3
case = GOLDEN[0]                      # 첫 케이스로 반복 실험
results = []
for i in range(K):
    ans = my_service(case["q"])
    v = judge(case["q"], ans, case["criteria"])
    results.append(v["score"] >= PASS)
    print(f"  시도 {i+1}: score={v['score']} {'✅' if results[-1] else '❌'}")
print(f"pass@{K} = {sum(results)}/{K} ({sum(results)/K:.0%})")
# 관찰: 3/3이면 안정, 1/3이면 '운으로 통과'하던 케이스 — 프롬프트를 다듬을 신호

  시도 1: score=5 ✅
  시도 2: score=5 ✅
  시도 3: score=4 ✅
pass@3 = 3/3 (100%)


## STEP 6 — 심판 검증: 사람 라벨과 얼마나 일치하나

강의의 3중 안전장치 ③ — 심판(LLM-as-judge)도 검증 대상입니다. 사람이 직접 라벨을 달고 **일치율(agreement)** 을 잽니다.

In [11]:
# 사람이 직접 판정한 라벨 (여러분이 답을 읽고 True/False로 바꿔보세요)
human_labels = {}
agree = 0
for case in GOLDEN[:3]:
    ans = my_service(case["q"])
    v = judge(case["q"], ans, case["criteria"])
    print(f"\nQ: {case['q']}\nA: {ans[:100]}...\n심판: {v['score']}점 — {v['reason'][:60]}")
    human = input("사람 판정 — 통과인가요? (y/n) ").strip().lower() == "y"
    human_labels[case["q"]] = human
    if human == (v["score"] >= PASS): agree += 1
print(f"\n심판↔사람 일치율: {agree}/3 — 낮으면 루브릭을 다듬어라 (심판을 맹신하지 말 것)")


Q: 강남에서 매운 점심 추천해줘
A: 강남에 있는 "매워서 미안해"를 추천합니다. 다양한 매운 noodle 요리를 제공하며, 매운 맛이 일품입니다. 분위기도 좋고, 점심에 가볍게 즐기기 좋은 메뉴가 많아 만족할 수 있...
심판: 5점 — 강남의 매운 음식점 하나를 구체적으로 추천하고, 음식의 특징과 분위기에 대한 이유를 잘 설명했다.

Q: 2만원 이하로 먹을 만한 곳 알려줘
A: 1. **떡볶이 집** - 매콤하고 쫄깃한 떡볶이가 2만원 이하로 푸짐하게 즐길 수 있습니다.

2. **김밥집** - 다양한 재료로 만든 김밥과 미소된장국 조합이 인기입니다. 가...
심판: 5점 — 2만원 이하의 예산을 잘 고려한 다양한 추천을 제시했다.

Q: 홍대에서 순한 음식 먹고 싶어
A: 홍대에서는 "한동네"를 추천합니다. 이곳은 비건 음식과 순한 한식 메뉴가 다양하게 있어 건강하게 식사할 수 있습니다. 분위기도 아늑하고 편안해요....
심판: 5점 — 홍대에서 순한 음식 옵션을 명확히 추천하였으며, 추가적인 정보도 제공했다.

심판↔사람 일치율: 3/3 — 낮으면 루브릭을 다듬어라 (심판을 맹신하지 말 것)


## STEP 7 — ⭐ 내 MVP에 적용 (실패 케이스로 회귀 잡기)

위 구조(골든셋 → judge → 통과율)는 **그대로** 당신 팀 MVP에 옮길 수 있습니다.
`my_service` 를 팀 기능으로 바꾸고, **실제로 틀렸던 케이스**를 골든셋에 추가해 보세요 — 통과율이 떨어지는 걸 눈으로 확인합니다.

⌨️ **터미널 Codex:** `codex "우리 MVP의 실패 사례 3개를 골든셋 케이스로 추가하고 평가를 다시 실행해줘"`


In [ ]:
# 팀별 작성: 우리 MVP에서 '실제로 틀렸던' 케이스를 추가하세요 (입력 → 기준)
MY_CASES_TODO = [
  # {"q": "...", "criteria": "..."},
  # {"q": "...", "criteria": "..."},
]

if MY_CASES_TODO:
    after = run_eval(GOLDEN + MY_CASES_TODO)
    print(f"\n변화: {baseline:.0f}% → {after:.0f}%  (떨어졌다면 골든셋이 회귀를 잡은 것!)")
else:
    print("⬜ MY_CASES_TODO를 채운 뒤 다시 실행하세요.")
    print("→ 실패 케이스를 넣으면 통과율이 떨어집니다 = 살아있는 회귀 테스트의 시작")

## 정리

- **평가 = 골든셋 + LLM-as-judge + 통과율.** ‘되는 것 같다’를 **숫자로 증명**할 수 있게 됐습니다.
- 다음 차시(16): 평가를 통과한 기능을 **안정적으로 배포·서빙(운영)** → 이후 관찰(17)·개선(18)으로 이어집니다.
